### Mount Google Drive

First, we need to mount your Google Drive to access the 'PDFs' directory. You'll be prompted to authorize this connection.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!pip install openai faiss-cpu langchain-text-splitters langchain-community pypdf

In [9]:
import re
import openai
import faiss
import numpy as np
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_534/2245103467.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
client = openai.OpenAI(api_key=userdata.get('openai-key'))

### Helper function to load PDF documents

This function will take the path to a PDF file, load it using `PyPDFLoader`, and return the document.

In [11]:
def load_pdf_document(pdf_path: str):
    """
    Loads a PDF document from the given path using PyPDFLoader.

    Args:
        pdf_path (str): The full path to the PDF file.

    Returns:
        list: A list of Document objects, where each object represents a page in the PDF.
    """
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        print(f"Successfully loaded {len(documents)} pages from {pdf_path}")
        return documents
    except Exception as e:
        print(f"Error loading PDF from {pdf_path}: {e}")
        return None

# Example usage (assuming PDFs are in '/content/drive/My Drive/PDFs/')
# Replace 'your_pdf_file.pdf' with the actual name of one of your PDF files
# pdf_file_path = '/content/drive/My Drive/PDFs/your_pdf_file.pdf'
# loaded_doc = load_pdf_document(pdf_file_path)
# if loaded_doc:
#     print(f"First page content: {loaded_doc[0].page_content[:200]}...")

In [12]:
pdf_names = ['Riipinen.pdf', 'Tsimpidi.pdf', 'Milousis_2024.pdf', 'Milousis_2025.pdf', 'Milousis_2026.pdf']
pdfs_directory = '/content/drive/My Drive/PDFs/'

all_documents = []

for pdf_name in pdf_names:
    pdf_path = f"{pdfs_directory}{pdf_name}"
    print(f"\nAttempting to load: {pdf_path}")
    documents = load_pdf_document(pdf_path)
    if documents:
        all_documents.extend(documents)

print(f"\nTotal pages loaded from all PDFs: {len(all_documents)}")
# You can inspect the first few pages of the loaded documents if needed
# for i, doc in enumerate(all_documents[:5]):
#     print(f"--- Document {i+1} ---")
#     print(doc.page_content[:200]) # Print first 200 characters of page content


Attempting to load: /content/drive/My Drive/PDFs/Riipinen.pdf
Successfully loaded 66 pages from /content/drive/My Drive/PDFs/Riipinen.pdf

Attempting to load: /content/drive/My Drive/PDFs/Tsimpidi.pdf
Successfully loaded 31 pages from /content/drive/My Drive/PDFs/Tsimpidi.pdf

Attempting to load: /content/drive/My Drive/PDFs/Milousis_2024.pdf
Successfully loaded 21 pages from /content/drive/My Drive/PDFs/Milousis_2024.pdf

Attempting to load: /content/drive/My Drive/PDFs/Milousis_2025.pdf
Successfully loaded 19 pages from /content/drive/My Drive/PDFs/Milousis_2025.pdf

Attempting to load: /content/drive/My Drive/PDFs/Milousis_2026.pdf
Successfully loaded 35 pages from /content/drive/My Drive/PDFs/Milousis_2026.pdf

Total pages loaded from all PDFs: 172


### 1. Helper function for basic text cleaning

In [13]:
def clean_text(text: str) -> str:
    """
    Performs basic text cleaning:
    - Lowercasing
    - Removal of consecutive tabs
    - Replacement of whitespace sequences with a single space
    """
    text = text.lower() # Lowercase the text
    text = re.sub(r'\t+', ' ', text) # Replace consecutive tabs with a single space
    text = re.sub(r'\s+', ' ', text) # Replace any sequence of whitespace characters with a single space
    return text.strip()

# Example usage:
# cleaned_example = clean_text("  This\t is   a  TEST\ttext with   various\twhitespaces. ")
# print(f"Original: '  This\t is   a  TEST\ttext with   various\twhitespaces. '")
# print(f"Cleaned: '{cleaned_example}'")

### 2. Helper function for splitting documents into chunks

In [14]:
def split_docs(documents: list, chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    Splits a list of documents into smaller chunks.

    Args:
        documents (list): A list of langchain Document objects.
        chunk_size (int): The maximum size of each chunk.
        chunk_overlap (int): The number of characters to overlap between chunks.

    Returns:
        list: A list of langchain Document objects representing the chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
    return chunks

# Note: This function expects `documents` to be a list of Langchain Document objects.

### 3. Helper function to create embeddings

In [15]:
def create_embeddings(chunks: list):
    """
    Creates embeddings for a list of text chunks using OpenAI's 'text-embedding-3-small' model.

    Args:
        chunks (list): A list of langchain Document objects, where each object's page_content is a chunk.

    Returns:
        list: A list of numpy arrays, where each array is the embedding for a corresponding chunk.
    """
    texts = [chunk.page_content for chunk in chunks]
    print(f"Creating embeddings for {len(texts)} chunks...")
    response = client.embeddings.create(input=texts, model="text-embedding-3-small")
    embeddings = [np.array(embedding_data.embedding) for embedding_data in response.data]
    print("Embeddings created.")
    return embeddings

### 4. Helper function to build a vector database (FAISS)

In [16]:
def build_vector_db(chunks: list, embeddings: list):
    """
    Builds a FAISS IndexFlatL2 vector database from embeddings and associates them with chunks.

    Args:
        chunks (list): A list of langchain Document objects.
        embeddings (list): A list of numpy arrays, corresponding to the embeddings of the chunks.

    Returns:
        tuple: A tuple containing the FAISS index and the list of chunks.
    """
    if not embeddings:
        print("No embeddings to build the vector database.")
        return None, []

    # Ensure embeddings are in a 2D array format for FAISS
    embeddings_array = np.array(embeddings).astype('float32')
    dimension = embeddings_array.shape[1]

    index = faiss.IndexFlatL2(dimension) # L2 distance for similarity
    index.add(embeddings_array)

    print(f"FAISS index built with {index.ntotal} vectors.")
    return index, chunks

### 5. Helper function to search for similar chunks

In [17]:
def search_chunks(query: str, index, chunks: list, k: int = 3):
    """
    Searches the FAISS index for relevant chunks based on a query.

    Args:
        query (str): The user's query string.
        index: The FAISS index.
        chunks (list): The list of langchain Document objects corresponding to the index.
        k (int): The number of top-k relevant chunks to retrieve.

    Returns:
        list: A list of the most relevant langchain Document objects.
    """
    query_embedding_response = client.embeddings.create(input=[query], model="text-embedding-3-small")
    query_embedding = np.array(query_embedding_response.data[0].embedding).astype('float32').reshape(1, -1)

    distances, indices = index.search(query_embedding, k) # Search for top-k similar vectors

    relevant_chunks = [chunks[i] for i in indices[0]]
    print(f"Found {len(relevant_chunks)} relevant chunks for the query.")
    return relevant_chunks

### 6. Function to generate answer using OpenAI GPT-4o-mini

In [18]:
def generate_answer(query, context_chunks):
    """
    Generates an answer to a query using the provided context and OpenAI's GPT-4o-mini.

    Args:
        query (str): The user's question.
        context_chunks (list): A list of langchain Document objects to use as context.

    Returns:
        str: The generated answer.
    """
    # Combine the text from the retrieved Documents
    context = "\n\n".join(doc.page_content for doc in context_chunks)

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer based on the context provided. If the answer is not available in the context, politely state that you cannot provide an answer based on the given information."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

### 7. Building the RAG system

In [23]:
def build_rag(documents: list):
    """
    Builds a Retrieval-Augmented Generation (RAG) system from a list of documents.

    Args:
        documents (list): A list of langchain Document objects.

    Returns:
        tuple: A tuple containing the FAISS index and the list of processed chunks.
    """
    print(f"Building RAG system with {len(documents)} initial documents...")

    if not documents:
        print("No documents provided to build the RAG system.")
        return None, []

    # 1. Clean each page (content of each Document)
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)

    # 2. Split into chunks (Documents with metadata)
    chunks = split_docs(documents)

    # 3. Create embeddings for each chunk
    embeddings = create_embeddings(chunks)

    # 4. Build vector DB (FAISS) with chunks + embeddings
    index, processed_chunks = build_vector_db(chunks, embeddings)

    if index is None:
        print("RAG system build failed.")
        return None, []

    print(f"Ready! Created {len(processed_chunks)} chunks.")
    return index, processed_chunks

### 8. Chat loop function

In [22]:
def chat(index, chunks):
    """
    Starts an interactive chat loop with the RAG system.

    Args:
        index: The FAISS index.
        chunks (list): The list of langchain Document objects corresponding to the index.
    """
    if index is None or not chunks:
        print("RAG system not properly initialized. Cannot start chat.")
        return

    print("Chat started. Type 'quit' to exit.")
    while True:
        query = input("Ask a question: ")
        if query.lower() == 'quit':
            break

        # Search for relevant chunks
        relevant_chunks = search_chunks(query, index, chunks)

        # Generate answer
        answer = generate_answer(query, relevant_chunks)

        # Format the answer to break lines after every 30 words
        words = answer.split()
        formatted_answer_lines = []
        for i in range(0, len(words), 30):
            formatted_answer_lines.append(' '.join(words[i:i+30]))
        formatted_answer = '\n'.join(formatted_answer_lines)

        print(f"\nAnswer:\n{formatted_answer}\n")

        # Show which chunks + metadata were used
        print("Context provided to the model (top 3 relevant chunks):")
        for i, doc in enumerate(relevant_chunks, 1):
            meta = doc.metadata
            # Assuming 'page' is a common metadata key, adjust if needed.
            page_info = meta.get('page', 'unknown') if 'page' in meta else 'unknown'
            print(f"--- Chunk {i} (Source: {meta.get('source', 'unknown_source')}, Page: {page_info}):")
            print(doc.page_content[:300], "...\n")  # show first 300 chars for readability

### 9. Functions to Save and Load RAG System

In [24]:
import pickle
import os

def save_rag_system(index, chunks, base_name: str = "rag_system"):
    """
    Saves the FAISS index and associated chunks to disk.

    Args:
        index: The FAISS index to save.
        chunks (list): The list of langchain Document objects to save.
        base_name (str): The base name for the saved files (e.g., 'rag_system').
    """
    index_path = f"{base_name}.faiss"
    chunks_path = f"{base_name}_chunks.pkl"

    try:
        faiss.write_index(index, index_path)
        with open(chunks_path, "wb") as f:
            pickle.dump(chunks, f)
        print(f"RAG system saved: {index_path} and {chunks_path}")
    except Exception as e:
        print(f"Error saving RAG system: {e}")

def load_rag_system(base_name: str = "rag_system"):
    """
    Loads the FAISS index and associated chunks from disk.

    Args:
        base_name (str): The base name for the saved files.

    Returns:
        tuple: A tuple containing the loaded FAISS index and the list of chunks,
               or (None, None) if loading fails.
    """
    index_path = f"{base_name}.faiss"
    chunks_path = f"{base_name}_chunks.pkl"

    if not os.path.exists(index_path) or not os.path.exists(chunks_path):
        print(f"Saved RAG system not found at {base_name}.faiss or {base_name}_chunks.pkl")
        return None, None

    try:
        index = faiss.read_index(index_path)
        with open(chunks_path, "rb") as f:
            chunks = pickle.load(f)
        print(f"RAG system loaded: {index_path} and {chunks_path}")
        return index, chunks
    except Exception as e:
        print(f"Error loading RAG system: {e}")
        return None, None

### Build and Save RAG System with All 5 Documents

In [25]:
# The 'all_documents' variable already contains all documents loaded from the 5 PDFs.
# It was generated by the cell that executed `for pdf_name in pdf_names: ...`

print("--- Building RAG system with all 5 documents ---")
all_pdfs_index, all_pdfs_chunks = build_rag(all_documents)

# Save the RAG system
if all_pdfs_index is not None and all_pdfs_chunks:
    save_rag_system(all_pdfs_index, all_pdfs_chunks, base_name="full_rag_system")
    print("\nRAG system built and saved. You can now load it in a new session.")
else:
    print("Failed to build RAG system for all PDFs.")


--- Building RAG system with all 5 documents ---
Building RAG system with 172 initial documents...
Split 172 documents into 1130 chunks.
Creating embeddings for 1130 chunks...
Embeddings created.
FAISS index built with 1130 vectors.
Ready! Created 1130 chunks.
RAG system saved: full_rag_system.faiss and full_rag_system_chunks.pkl

RAG system built and saved. You can now load it in a new session.


### 10. Helper function to move files to Google Drive

In [28]:
import shutil

def move_files_to_drive(file_paths: list, destination_folder_name: str):
    """
    Moves a list of files from the current directory to a specified Google Drive folder.

    Args:
        file_paths (list): A list of strings, where each string is the path to a file to move.
        destination_folder_name (str): The name of the folder in Google Drive's 'My Drive'
                                       to which files will be moved. If it doesn't exist, it will be created.
    """
    drive_path = f'/content/drive/My Drive/{destination_folder_name}'

    # Create the destination folder if it doesn't exist
    os.makedirs(drive_path, exist_ok=True)
    print(f"Ensured Google Drive folder exists: {drive_path}")

    for file_path in file_paths:
        if os.path.exists(file_path):
            try:
                shutil.move(file_path, drive_path)
                print(f"Moved '{file_path}' to '{drive_path}'")
            except Exception as e:
                print(f"Error moving '{file_path}': {e}")
        else:
            print(f"File not found: '{file_path}'. Skipping.")



### Move the RAG system files to Google Drive

In [29]:
# Define the files to move
rag_system_files = [
    'full_rag_system.faiss',
    'full_rag_system_chunks.pkl'
]

# Define the destination folder in Google Drive
drive_rag_folder = 'My_RAG_Systems'

# Move the files
move_files_to_drive(rag_system_files, drive_rag_folder)

print(f"\nThe RAG system files are now permanently stored in your Google Drive at '/content/drive/My Drive/{drive_rag_folder}'.")

Ensured Google Drive folder exists: /content/drive/My Drive/My_RAG_Systems
Moved 'full_rag_system.faiss' to '/content/drive/My Drive/My_RAG_Systems'
Moved 'full_rag_system_chunks.pkl' to '/content/drive/My Drive/My_RAG_Systems'

The RAG system files are now permanently stored in your Google Drive at '/content/drive/My Drive/My_RAG_Systems'.


### Load and Chat with the Full RAG System

In [27]:
print("--- Attempting to load previously saved RAG system ---")
loaded_index, loaded_chunks = load_rag_system(base_name="full_rag_system")

# Start the chat if the RAG system was successfully loaded
if loaded_index is not None and loaded_chunks:
    print("\n--- Starting chat with the full RAG system ---")
    chat(loaded_index, loaded_chunks)
else:
    print("Failed to load RAG system. Cannot start chat.")

--- Attempting to load previously saved RAG system ---
RAG system loaded: full_rag_system.faiss and full_rag_system_chunks.pkl

--- Starting chat with the full RAG system ---
Chat started. Type 'quit' to exit.
Ask a question: What are nitrate aerosols ?
Found 3 relevant chunks for the query.

Answer:
Nitrate aerosols are tiny particles in the atmosphere that contain nitrate compounds. They are significant because they can influence atmospheric chemistry through heterogeneous reactions with dust and sea salt, leading
to more acidic conditions in aerosols. Additionally, nitrate aerosols can affect climate through direct interactions with sunlight and other atmospheric processes. Their formation is sensitive to the levels of their
precursors, and they are inherently semi-volatile, meaning their distribution between gas and particle phases is complex and influenced by factors such as humidity, temperature, and atmospheric acidity.

Context provided to the model (top 3 relevant chunks):
---

## RAG System Evaluation

To assess the performance of our RAG system, we will implement several key evaluation metrics. Since `ragas` encountered dependency issues, we will define custom functions to calculate these metrics using direct LLM calls via the `openai` client. This approach allows us to control the evaluation process and understand each step.

### Metrics to be evaluated:

1.  **Context Relevancy**: How relevant are the retrieved document chunks to the user's query?
2.  **Answer Faithfulness**: Is the generated answer factually consistent with the retrieved context?
3.  **Answer Relevancy**: Does the generated answer directly and comprehensively address the user's question?
4.  **Context Recall (Simplified)**: Is the original document chunk (from which the evaluation question was derived) successfully retrieved by the RAG system?

### 1. Generating a Synthetic Evaluation Dataset

We need a set of questions, their ground-truth answers, the contexts retrieved by our RAG system, and the answers generated by our RAG system. We'll create this synthetically by taking existing document chunks, asking an LLM to generate a question and a concise answer *from that chunk*, and then running that question through our RAG system to get its output.

In [37]:
import random
import json
import re # For parsing LLM responses

def generate_eval_dataset(chunks_list: list, num_samples: int = 10):
    """
    Generates a synthetic evaluation dataset for RAG systems.

    Args:
        chunks_list (list): A list of langchain Document objects (chunks) from which to generate questions.
        num_samples (int): The number of questions to generate.

    Returns:
        list: A list of dictionaries, each containing 'question', 'ground_truth_answer', 'contexts', 'answer', and 'original_chunk_content'.
    """
    eval_data = []
    print(f"Generating {num_samples} evaluation samples...")

    # Ensure there are enough chunks to sample
    if len(chunks_list) < num_samples:
        num_samples = len(chunks_list)
        print(f"Warning: Not enough chunks for {num_samples} samples, reducing to {len(chunks_list)}.")

    # Randomly sample chunks to create questions from
    sampled_chunks = random.sample(chunks_list, num_samples)

    for i, chunk in enumerate(sampled_chunks):
        print(f"Processing sample {i+1}/{num_samples}...")
        context_text = chunk.page_content

        # Use LLM to generate question and ground truth answer from the chunk
        qa_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant. Generate a single question and a concise answer based ONLY on the provided text. The question should be answerable directly from the text. Respond in JSON format: { \"question\": \"your question\", \"answer\": \"your answer\" }"},
                {"role": "user", "content": f"Text: {context_text}"}
            ],
            response_format={"type": "json_object"},
            temperature=0.7
        )
        try:
            qa_pair = json.loads(qa_response.choices[0].message.content)
            question = qa_pair['question']
            ground_truth_answer = qa_pair['answer']
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from LLM for QA pair: {e}. Skipping sample.")
            continue

        # Use our RAG system to get retrieved contexts and generated answer for the question
        # Assuming 'all_pdfs_index' and 'all_pdfs_chunks' are available from previous steps
        retrieved_contexts_docs = search_chunks(question, all_pdfs_index, all_pdfs_chunks, k=3)
        generated_answer = generate_answer(question, retrieved_contexts_docs)

        eval_data.append({
            "question": question,
            "ground_truth_answer": ground_truth_answer,
            "contexts": [doc.page_content for doc in retrieved_contexts_docs],
            "answer": generated_answer,
            "original_chunk_content": context_text # Store original chunk content for context recall
        })
    print("Evaluation dataset generation complete!")
    return eval_data

# Generate the evaluation dataset (using a small number of samples for demonstration)
# Increase `num_samples` for a more robust evaluation.
# Ensure 'all_pdfs_chunks' is available from the RAG system build step.
rag_eval_dataset = generate_eval_dataset(all_pdfs_chunks, num_samples=5)

print("\nGenerated Evaluation Samples:")
for i, sample in enumerate(rag_eval_dataset):
    print(f"--- Sample {i+1} ---")
    print(f"Q: {sample['question']}")
    print(f"GT Answer: {sample['ground_truth_answer']}")
    print(f"RAG Answer: {sample['answer']}")
    print(f"Retrieved Contexts (first chunk): {sample['contexts'][0][:200]}...")


Generating 5 evaluation samples...
Processing sample 1/5...
Found 3 relevant chunks for the query.
Processing sample 2/5...
Found 3 relevant chunks for the query.
Processing sample 3/5...
Found 3 relevant chunks for the query.
Processing sample 4/5...
Found 3 relevant chunks for the query.
Processing sample 5/5...
Found 3 relevant chunks for the query.
Evaluation dataset generation complete!

Generated Evaluation Samples:
--- Sample 1 ---
Q: What percentage underprediction does htapv3 show at rural sites in Europe compared to pm 1 nitrate concentrations from ams field campaigns?
GT Answer: Nearly 20% underprediction.
RAG Answer: htapv3 shows nearly 20% underprediction at rural sites in Europe compared to pm 1 nitrate concentrations from ams field campaigns.
Retrieved Contexts (first chunk): simulation yields the high- est average pm2.5 nitrate concentrations among all sensitivity runs (fig. 7a). the emission behavior in this region is sim- ilar to the u.s. and europe, with no x and nh 

### 2. Custom Metric Calculation Functions

Here we define functions to calculate each of the desired evaluation metrics using LLM calls.

In [40]:
def llm_score_evaluation(prompt: str, model: str = "gpt-4o-mini") -> str:
    """
    Helper function to get LLM response for evaluation scoring.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt + "\nRespond with only the rating number."}
        ],
        temperature=0.0 # Make it deterministic for scoring
    )
    return response.choices[0].message.content


def calculate_context_relevancy(eval_data: list) -> float:
    """
    Calculates the average context relevancy score using an LLM.
    Scores each retrieved chunk's relevancy to the question on a scale of 1 to 5.
    """
    scores = []
    print("Calculating Context Relevancy...")
    for i, item in enumerate(eval_data):
        question = item["question"]
        contexts = item["contexts"]
        if not contexts:
            scores.append(1) # If no context, it's not relevant to anything, lowest score
            continue

        chunk_scores = []
        for context_chunk in contexts:
            prompt = f"""
            Given the following question and context, rate the context's relevance to the question on a scale of 1 to 5.
            1: Not relevant at all
            2: Slightly relevant
            3: Moderately relevant
            4: Highly relevant
            5: Directly answers the question or provides essential information

            Question: {question}
            Context: {context_chunk}
            """
            response = llm_score_evaluation(prompt)
            try:
                rating_match = re.search(r'(\d+)', response) # Modified regex to extract just the number
                if rating_match:
                    rating = int(rating_match.group(1))
                    chunk_scores.append(rating)
                else:
                    print(f"Warning: Could not parse rating for context relevancy in sample {i}. Response: {response}. Defaulting to 1.")
                    chunk_scores.append(1) # Default to lowest if parsing fails
            except (AttributeError, ValueError):
                print(f"Error parsing rating for context relevancy in sample {i}. Response: {response}. Defaulting to 1.")
                chunk_scores.append(1) # Default to lowest if parsing fails

        if chunk_scores:
            scores.append(sum(chunk_scores) / len(chunk_scores))
        else:
            scores.append(1) # Default if no valid chunk scores were obtained

    return sum(scores) / len(scores) if scores else 0

def calculate_answer_faithfulness(eval_data: list) -> float:
    """
    Calculates the average answer faithfulness score using an LLM.
    Determines if the generated answer is fully supported by the provided context (binary score).
    """
    scores = []
    print("Calculating Answer Faithfulness...")
    for i, item in enumerate(eval_data):
        answer = item["answer"]
        contexts = item["contexts"]
        if not contexts:
            scores.append(0) # Cannot be faithful if no context
            continue

        context_str = "\n\n".join(contexts)

        prompt = f"""
        Given the following generated answer and context, determine if the generated answer is fully supported by the provided context.
        Respond with 'YES' if every statement in the answer can be directly inferred from the context. Respond with 'NO' otherwise (if it contains information not found in the context or contradicts the context).

        Generated Answer: {answer}
        Context: {context_str}
        """
        response = llm_score_evaluation(prompt)
        if "yes" in response.lower():
            scores.append(1)
        else:
            scores.append(0)
    return sum(scores) / len(scores) if scores else 0

def calculate_answer_relevancy(eval_data: list) -> float:
    """
    Calculates the average answer relevancy score using an LLM.
    Rates how relevant the generated answer is to the question on a scale of 1 to 5.
    """
    scores = []
    print("Calculating Answer Relevancy...")
    for i, item in enumerate(eval_data):
        question = item["question"]
        answer = item["answer"]

        prompt = f"""
        Given the following question and answer, rate how relevant the answer is to the question on a scale of 1 to 5.
        1: Not relevant
        2: Slightly relevant
        3: Moderately relevant
        4: Highly relevant
        5: Directly and comprehensively answers the question

        Question: {question}
        Answer: {answer}
        """
        response = llm_score_evaluation(prompt)
        try:
            rating_match = re.search(r'(\d+)', response) # Modified regex to extract just the number
            if rating_match:
                rating = int(rating_match.group(1))
                scores.append(rating)
            else:
                print(f"Warning: Could not parse rating for answer relevancy in sample {i}. Response: {response}. Defaulting to 1.")
                scores.append(1) # Default to lowest if parsing fails
        except (AttributeError, ValueError):
            print(f"Error parsing rating for answer relevancy in sample {i}. Response: {response}. Defaulting to 1.")
            scores.append(1) # Default to lowest if parsing fails
    return sum(scores) / len(scores) if scores else 0

def calculate_context_recall(eval_data: list) -> float:
    """
    Calculates the context recall (simplified) by checking if the original chunk content
    (from which the question was generated) is present in the retrieved contexts.
    """
    hits = 0
    print("Calculating Context Recall...")
    for item in eval_data:
        original_chunk_content = item["original_chunk_content"]
        retrieved_contexts = item["contexts"]

        # Check if the content of the original chunk is present in any of the retrieved contexts
        # A simple substring check is used here for simplicity. For more robustness,
        # one might consider embedding similarity or more advanced text comparison.
        if any(original_chunk_content in rc for rc in retrieved_contexts):
            hits += 1
    return hits / len(eval_data) if eval_data else 0

### 3. Run Evaluation and Display Results

Now we will execute the evaluation functions and display the computed metrics.

In [41]:
print("--- Starting RAG System Evaluation ---")

# Ensure 'rag_eval_dataset' is populated from the previous step
if not rag_eval_dataset:
    print("Evaluation dataset is empty. Please generate it first.")
else:
    # Calculate metrics
    context_relevancy_score = calculate_context_relevancy(rag_eval_dataset)
    answer_faithfulness_score = calculate_answer_faithfulness(rag_eval_dataset)
    answer_relevancy_score = calculate_answer_relevancy(rag_eval_dataset)
    context_recall_score = calculate_context_recall(rag_eval_dataset)

    print("\n--- RAG Evaluation Results ---")
    print(f"Average Context Relevancy: {context_relevancy_score:.2f} (Scale: 1-5)")
    print(f"Average Answer Faithfulness: {answer_faithfulness_score:.2f} (Binary: 0=No, 1=Yes)")
    print(f"Average Answer Relevancy: {answer_relevancy_score:.2f} (Scale: 1-5)")
    print(f"Average Context Recall: {context_recall_score:.2f} (Binary: 0=No, 1=Yes)")

    print("\nEvaluation complete!")

--- Starting RAG System Evaluation ---
Calculating Context Relevancy...
Calculating Answer Faithfulness...
Calculating Answer Relevancy...
Calculating Context Recall...

--- RAG Evaluation Results ---
Average Context Relevancy: 2.93 (Scale: 1-5)
Average Answer Faithfulness: 0.40 (Binary: 0=No, 1=Yes)
Average Answer Relevancy: 5.00 (Scale: 1-5)
Average Context Recall: 0.80 (Binary: 0=No, 1=Yes)

Evaluation complete!


### Demonstration with 'Riipinen.pdf'

In [ ]:
# # Define the path to 'Riipinen.pdf'
# riipinen_pdf_path = '/content/drive/My Drive/PDFs/Riipinen.pdf'

# # Build the RAG system for Riipinen.pdf
# riipinen_index, riipinen_chunks = build_rag(riipinen_pdf_path)

# # Start the chat if the RAG system was successfully built
# if riipinen_index is not None and riipinen_chunks:
#     chat(riipinen_index, riipinen_chunks)
# else:
#     print("Failed to build RAG system for Riipinen.pdf. Cannot start chat.")